# Análise complementar dos perfis de clientes

Autor: Arthur Schmitz Donato Cavalcanti

Análise dos grupos já definidos pelo K-Means, sem alterar os dados ou usar churn para formar os grupos.

In [1]:
from pathlib import Path
import pandas as pd

pasta = Path("reports/tabelas")
if not pasta.exists():
    pasta = Path("../reports/tabelas")

clientes = pd.read_csv(pasta / "clientes_clusters.csv")
tamanhos = pd.read_csv(pasta / "perfil_tamanho.csv")

display(clientes.head())
display(tamanhos)

,row_id,call_failure,complains,subscription_length,charge_amount,seconds_of_use,frequency_of_use,frequency_of_sms,distinct_called_numbers,tariff_plan,status,age,customer_value,cluster,churn,conjunto
0,0,8,0,38,0,4370,71,5,17,0,0,30,197.640,1,0,teste
1,1,0,0,39,0,318,5,7,4,0,1,25,46.035,1,0,treino
2,2,10,0,37,0,2453,60,359,24,0,0,30,1536.520,0,0,treino
3,3,10,0,38,0,4198,66,1,35,0,0,15,240.020,1,0,treino
4,4,3,0,38,0,2393,58,2,33,0,0,15,145.805,1,0,treino


,grupo,perfil,clientes,participacao,cancelaram,taxa_churn,taxa_churn_treino,taxa_churn_teste
0,0,Intensivo em mensagens,575,0.182540,5,0.008696,0.006536,0.017241
1,1,Baixo engajamento,1875,0.595238,486,0.259200,0.260549,0.253927
2,2,Intensivo em voz,700,0.222222,4,0.005714,0.005310,0.007407


## 1. Contagens e taxas por grupo
Calculamos o total de clientes, a participação e a taxa de cancelamento de cada cluster.

In [2]:
resumo = clientes.groupby("cluster").agg(
    clientes=("churn", "size"),
    cancelamentos=("churn", "sum"),
    taxa_churn=("churn", "mean")
)

resumo["participacao"] = resumo["clientes"] / len(clientes) * 100
resumo["taxa_churn"] *= 100

display(resumo.round(2))

,clientes,cancelamentos,taxa_churn,participacao
cluster,,,,
0,575,5,0.87,18.25
1,1875,486,25.92,59.52
2,700,4,0.57,22.22


In [3]:
por_conjunto = clientes.groupby(["conjunto", "cluster"]).agg(
    clientes=("churn", "size"),
    cancelamentos=("churn", "sum"),
    taxa_churn=("churn", "mean")
)

por_conjunto["participacao"] = (
    por_conjunto["clientes"]
    / por_conjunto.groupby(level="conjunto")["clientes"].transform("sum")
    * 100
)
por_conjunto["taxa_churn"] *= 100

display(por_conjunto.round(2))

clientes  cancelamentos  taxa_churn  participacao
conjunto cluster                                                   
teste    0             116              2        1.72         18.33
         1             382             97       25.39         60.35
         2             135              1        0.74         21.33
treino   0             459              3        0.65         18.24
         1            1493            389       26.05         59.32
         2             565              3        0.53         22.45

## 2. Características dos grupos
Calculamos média, mediana, desvio-padrão e quartis dos atributos, excluindo identificador, cluster, churn e conjunto.


In [4]:
atributos = clientes.drop(
    columns=["row_id", "cluster", "churn", "conjunto"]
).columns

estatisticas = (
    clientes.groupby("cluster")[atributos]
    .describe()
    .stack(level=0)
)

display(estatisticas[["mean", "50%", "std", "25%", "75%"]].round(2))

/tmp/ipykernel_18707/2694471526.py:8: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack(level=0)


mean       50%      std      25%  \
cluster                                                                 
0       age                         28.52     30.00     3.82    25.00   
        call_failure                 5.25      5.00     4.57     1.00   
        charge_amount                1.03      1.00     1.14     0.00   
        complains                    0.02      0.00     0.13     0.00   
        customer_value            1319.19   1186.04   425.75   951.75   
        distinct_called_numbers     22.47     22.00    12.01    12.00   
        frequency_of_sms           277.83    261.00   105.02   191.00   
        frequency_of_use            65.20     54.00    34.88    38.00   
        seconds_of_use            4256.45   3375.00  2237.30  2475.00   
        status                       0.00      0.00     0.00     0.00   
        subscription_length         32.60     35.00     9.45    31.00   
        tariff_plan                  0.21      0.00     0.41     0.00   
1       age                         30.88     30.00     8.83    25.00   
        call_failure                 5.58      5.00     5.81     0.00   
        charge_amount                0.40      0.00     0.78     0.00   
        complains                    0.11      0.00     0.32     0.00   
        customer_value             148.28    135.58   115.17    66.09   
        distinct_called_numbers     16.57     15.00    13.46     6.00   
        frequency_of_sms            15.28     10.00    18.08     0.00   
        frequency_of_use            40.25     37.00    29.17    16.00   
        seconds_of_use            2230.88   1855.00  1759.77   825.00   
        status                       0.42      0.00     0.49     0.00   
        subscription_length         32.29     35.00     8.13    29.00   
        tariff_plan                  0.01      0.00     0.08     0.00   
2       age                         33.35     30.00    10.94    25.00   
        call_failure                15.06     15.00     7.67    10.00   
        charge_amount                2.31      2.00     2.26     1.00   
        complains                    0.03      0.00     0.16     0.00   
        customer_value             638.57    619.60   347.14   349.71   
        distinct_called_numbers     42.95     41.00    14.92    34.00   
        frequency_of_sms            60.13     45.50    56.61    11.00   
        frequency_of_use           151.19    153.00    50.75   106.00   
        seconds_of_use           10654.14  10365.00  3897.68  7102.75   
        status                       0.00      0.00     0.00     0.00   
        subscription_length         33.17     36.00     8.95    31.00   
        tariff_plan                  0.16      0.00     0.37     0.00   

                                      75%  
cluster                                    
0       age                         30.00  
        call_failure                 8.50  
        charge_amount                2.00  
        complains                    0.00  
        customer_value            1628.70  
        distinct_called_numbers     30.00  
        frequency_of_sms           362.50  
        frequency_of_use            90.00  
        seconds_of_use            6615.00  
        status                       0.00  
        subscription_length         38.00  
        tariff_plan                  0.00  
1       age                         30.00  
        call_failure                 9.00  
        charge_amount                1.00  
        complains                    0.00  
        customer_value             209.08  
        distinct_called_numbers     23.00  
        frequency_of_sms            23.00  
        frequency_of_use            62.00  
        seconds_of_use            3319.00  
        status                       1.00  
        subscription_length         38.00  
        tariff_plan                  0.00  
2       age                         45.00  
        call_failure                20.00  
        charge_amount         

In [5]:
caracteristicas = pd.read_csv(pasta / "perfil_caracteristicas.csv")
display(caracteristicas)

,atributo,unidade,geral,Intensivo em mensagens,Baixo engajamento,Intensivo em voz
0,Falhas de chamada,falhas de chamada,7.63,5.25,5.58,15.06
1,Reclamação registrada,proporção com reclamação,0.08,0.02,0.11,0.03
2,Tempo de assinatura,meses,32.54,32.60,32.29,33.17
3,Faixa de cobrança,faixa ordinal (0–10),0.94,1.03,0.40,2.31
4,Segundos de uso,segundos,4472.46,4256.45,2230.88,10654.14
5,Chamadas realizadas,chamadas,69.46,65.20,40.25,151.19
6,Mensagens enviadas,mensagens,73.17,277.83,15.28,60.13
7,Contatos distintos,números distintos,23.51,22.47,16.57,42.95
8,Plano por contrato,proporção em contrato,0.08,0.21,0.01,0.16
9,Linha inativa,proporção inativa,0.25,0.00,0.42,0.00


## 3. Incerteza das taxas de churn
Calculamos intervalos de confiança de Wilson a 95%, pois os clusters 0 e 2 apresentam apenas 5 e 4 cancelamentos.

In [6]:
from statsmodels.stats.proportion import proportion_confint

intervalos = resumo[["clientes", "cancelamentos", "taxa_churn"]].copy()

inferior, superior = proportion_confint(
    count=intervalos["cancelamentos"],
    nobs=intervalos["clientes"],
    alpha=0.05,
    method="wilson"
)

intervalos["ic95_inferior"] = inferior * 100
intervalos["ic95_superior"] = superior * 100

display(intervalos.round(2))

,clientes,cancelamentos,taxa_churn,ic95_inferior,ic95_superior
cluster,,,,,
0,575,5,0.87,0.37,2.02
1,1875,486,25.92,23.99,27.95
2,700,4,0.57,0.22,1.46


Os clusters 0 e 2 possuem 575 e 700 clientes, mas apenas 5 e 4 cancelamentos. Suas taxas de churn são 0,87% e 0,57%, com intervalos de Wilson a 95% de 0,37%–2,02% e 0,22%–1,46%, respectivamente. Esses poucos eventos exigem cautela na comparação das taxas.

O cluster 1 apresenta 486 cancelamentos e taxa de 25,92%, com intervalo de 23,99%–27,95%. Os resultados indicam associação entre os perfis e o churn, sem demonstrar causalidade.

## 4. Estabilidade entre execuções
Analisamos as tabelas de sensibilidade produzidas por Artur Rio, sem executar novamente o K-Means.

In [7]:
for arquivo in ["sens_sementes.csv", "sens_escala.csv",
                "sens_atributos.csv", "revisao_k_perfis.csv"]:
    print(arquivo)
    display(pd.read_csv(pasta / arquivo))

sens_sementes.csv


,n_init,k,silhueta_media,silhueta_dp,silhueta_min,silhueta_max,inercia_media,inercia_dp,ari_medio,ari_minimo,particoes_distintas
0,1,2,0.304224,1.110223e-16,0.304224,0.304224,17087.846154,0.007313,1.000000,1.000000,2
1,1,3,0.292563,3.262990e-02,0.196321,0.306354,14195.167555,650.046568,0.822172,0.411737,14
2,1,4,0.253549,3.393063e-02,0.198231,0.312476,12560.130927,79.738217,0.600869,0.360769,20
3,1,5,0.229640,2.466833e-02,0.199118,0.281639,11319.698323,147.430104,0.568844,0.297621,20
4,1,6,0.239733,2.288837e-02,0.205287,0.277583,10175.154987,133.265196,0.613228,0.374438,20
5,20,2,0.304224,1.110223e-16,0.304224,0.304224,17087.846154,0.007313,1.000000,1.000000,2
6,20,3,0.306076,2.779273e-04,0.304966,0.306354,13872.276690,0.021150,0.997126,0.988481,8
7,20,4,0.202302,2.246789e-03,0.198506,0.204080,12468.012843,0.355028,0.953806,0.890238,20
8,20,5,0.250309,2.206341e-02,0.216249,0.264912,11191.823609,27.685287,0.777854,0.505636,20
9,20,6,0.234267,6.634274e-04,0.232350,0.234643,10037.638070,20.194083,0.964081,0.820434,20


sens_escala.csv


,escala,k,silhueta,inercia,menor_grupo,maior_grupo
0,standard,2,0.304224,1.708784e+04,866,1651
1,standard,3,0.306064,1.387226e+04,459,1493
2,standard,4,0.203857,1.246782e+04,356,880
3,standard,5,0.264912,1.117362e+04,310,1070
4,standard,6,0.234643,1.003124e+04,226,675
5,minmax,2,0.379176,1.223496e+03,616,1901
6,minmax,3,0.332867,9.991318e+02,578,1323
7,minmax,4,0.342678,8.400945e+02,418,1002
8,minmax,5,0.361780,7.013445e+02,200,962
9,minmax,6,0.394523,5.919579e+02,162,924


sens_atributos.csv


,conjunto,n_atributos,k,silhueta,menor_grupo
0,completo (12),12,2,0.304224,866
1,completo (12),12,3,0.306064,459
2,completo (12),12,4,0.203857,356
3,completo (12),12,5,0.264912,310
4,completo (12),12,6,0.234643,226
5,sem binarias (9),9,2,0.317381,853
6,sem binarias (9),9,3,0.318345,456
7,sem binarias (9),9,4,0.270267,324
8,sem binarias (9),9,5,0.278449,307
9,sem binarias (9),9,6,0.278690,156


revisao_k_perfis.csv


,k,cluster,clientes,silhueta_media,taxa_churn
0,2,0,1651,0.408848,0.237432
1,2,1,866,0.104762,0.003464
2,3,0,459,0.275446,0.006536
3,3,1,1493,0.379473,0.260549
4,3,2,565,0.136953,0.005310
5,4,0,444,0.254188,0.006757
6,4,1,837,0.106350,0.140980
7,4,2,356,0.149726,0.000000
8,4,3,880,0.293104,0.311364


### Interpretação

Para k=3, aumentar n_init de 1 para 20 elevou o ARI médio de 0,8222 para 0,9971 e o ARI mínimo de 0,4117 para 0,9885. Isso indica maior concordância entre as execuções, embora as partições não sejam todas idênticas.

A composição dos grupos é sensível à escala. Para k=3, o menor grupo contém 459 clientes com padronização, 578 com MinMax e 263 sem escalonamento. As silhuetas dessas configurações são calculadas em espaços diferentes; um valor maior, isoladamente, não demonstra que uma escala produz perfis melhores.

A retirada de customer_value também altera os resultados: para k=3, a silhueta passa de 0,3061 para 0,2369. Sem esse atributo, k=2 apresenta silhueta de 0,3111. Essas medidas indicam sensibilidade à seleção de atributos, mas não mostram quais perfis se fundem.

Na revisão de k=3, os tamanhos 459, 1.493 e 565 correspondem ao conjunto de treino. A silhueta média do cluster 2 é 0,1370, inferior às dos clusters 0 e 1, de 0,2754 e 0,3795, indicando menor separação desse grupo no espaço utilizado.

A seção 7.1 do artigo relata que, sem customer_value, a separação entre os perfis intensivos em mensagens e em voz não se sustenta. As tabelas de sensibilidade mostram que, sem esse atributo, k=2 apresenta silhueta superior a k=3. Entretanto, essas tabelas resumidas não permitem verificar diretamente quais clientes mudam de grupo. Assim, a interpretação dos dois perfis engajados depende da seleção dos atributos e deve ser tratada com cautela.

## 5. Descrição complementar dos perfis

As descrições abaixo utilizam apenas os atributos de entrada. O churn não foi utilizado para definir os nomes dos grupos.

### Cluster 0 — Intensivo em mensagens

Este grupo apresenta a maior média de mensagens enviadas: 277,83, contra 15,28 no cluster 1 e 60,13 no cluster 2. A média de segundos de uso é de 4.256,45, inferior à do cluster 2 e superior à do cluster 1. O customer_value médio também é o maior entre os grupos, com 1.319,19. A predominância relativa de mensagens sustenta o nome “Intensivo em mensagens”.

### Cluster 1 — Baixo engajamento

Este grupo apresenta as menores médias de segundos de uso (2.230,88), frequência de uso (40,25) e frequência de mensagens (15,28). A média de números distintos contatados é de 16,57, também a menor entre os grupos. O customer_value médio é de 148,28, abaixo dos demais perfis. Esses indicadores sustentam o nome “Baixo engajamento”, entendido como menor utilização dos serviços observados na base.

### Cluster 2 — Intensivo em voz

Este grupo apresenta as maiores médias de segundos de uso (10.654,14) e frequência de uso (151,19). Também possui a maior média de números distintos contatados, de 42,95. A média de mensagens é de 60,13, inferior à do cluster 0. A maior utilização das chamadas sustenta o nome “Intensivo em voz”.

Essas descrições resumem médias dos grupos e não representam necessariamente cada cliente. As medianas, os quartis e os desvios-padrão apresentados na seção 2 devem ser considerados para avaliar a variação dentro de cada perfil.